# 🌍 Fine-Tuning com LoRA — google/mt5-large
### Dataset: Neurociência Cognitiva

Este notebook adapta o pipeline LoRA para o **mT5-Large** (1.2B parâmetros),
a versão **massivamente multilíngue** do T5, pré-treinada em 101 idiomas.

### ⚠️ Diferenças importantes em relação ao Flan-T5-XL

| Aspecto | Flan-T5-XL | mT5-Large |
|---|---|---|
| Pré-treinamento | Supervisionado (instruções) | **Apenas não supervisionado** (mC4) |
| Prefixo de instrução | Necessário e eficaz | **Não usar** — modelo não foi treinado com prefixos |
| Vocabulário | 32K tokens | **250K tokens** (101 idiomas) |
| Tokenizador | `AutoTokenizer` | **`AutoTokenizer` com `use_fast=False`** |
| Geração sem fine-tuning | Razoável | **Muito fraca** — requer fine-tuning para qualquer tarefa |


## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias:


In [ ]:
# Descomente e execute apenas uma vez
# sentencepiece é obrigatório para o MT5Tokenizer (vocabulário de 250K tokens)
#!pip install transformers datasets peft accelerate torch bitsandbytes sentencepiece -q


Importe os módulos que serão utilizados ao longo do processo.

> **Nota:** O `MT5Tokenizer` foi descontinuado em versões recentes do Transformers.
> Usamos `AutoTokenizer` com `use_fast=False` para carregar o tokenizador SentencePiece
> do mT5 com o comportamento correto para o vocabulário de 250K tokens.


In [ ]:
from datasets import load_dataset
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    # MT5Tokenizer foi descontinuado — usar AutoTokenizer com use_fast=False
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🤖 3. Carregar modelo e tokenizador

O mT5-Large tem **1.2B parâmetros** distribuídos entre encoder e decoder.

| Precisão | VRAM estimada (1.2B params) |
|---|---|
| float32 (sem quantização) | ~5 GB |
| float16 / bfloat16 | ~2.5 GB |
| 4-bit QLoRA | ~1–2 GB |

> **Por que `AutoTokenizer` com `use_fast=False`?**  
> O mT5 usa SentencePiece com 250K tokens cobrindo 101 idiomas.
> O `MT5Tokenizer` foi descontinuado em versões recentes do Transformers —
> `AutoTokenizer` com `use_fast=False` é a forma atual recomendada e evita `ImportError`.


In [ ]:
model_name = "MBZUAI/LaMini-Flan-T5-783M"   

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# use_fast=False — necessário para o tokenizador SentencePiece do mT5
# MT5Tokenizer foi descontinuado; AutoTokenizer é a forma atual recomendada
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Modelo carregado: {model_name}')
print(f'Parâmetros totais: {sum(p.numel() for p in base_model.parameters()) / 1e9:.2f}B')
print(f'Tamanho do vocabulário: {tokenizer.vocab_size:,} tokens')


## 📂 4. Carregar e preparar o dataset

### ⚠️ Não usar prefixo de instrução no mT5

O Flan-T5 foi treinado com prefixos como `"Responda sobre neurociência: "`
porque seu fine-tuning supervisionado incluía tarefas com esse formato.

O mT5, por outro lado, foi **pré-treinado apenas de forma não supervisionada**
no corpus mC4 — sem nenhuma tarefa de instrução. Adicionar um prefixo não traz
benefício e pode confundir o modelo durante o fine-tuning.

A entrada vai direto ao ponto:
```
input_ids → "O que é memória de trabalho?"
labels    → "A memória de trabalho é um sistema cognitivo..."
```


In [ ]:
TEST_INSTRUCTION = "Quais são os principais sintomas da demência que afetam a vida diária das pessoas idosas?"
PREFIX = "Responda sobre neurociência cognitiva: "

def convert_to_hf_format(example):
    """
    Converte o dataset para o formato do mT5.
    Sem prefixo de instrução — o mT5 não foi pré-treinado com esse padrão.
    """
    return {
        "input_text":  PREFIX + example["Instruction"],  # ← com prefixo
        "target_text": example["Output"]
    }

dataset = load_dataset(
    "json",
    data_files="data/processed/dataset_curado.jsonl"
)

dataset = dataset.map(convert_to_hf_format)
dataset = dataset["train"].train_test_split(test_size=0.2)

print(dataset)
print("\nExemplo de entrada:")
print(dataset["train"][0]["input_text"])


Map: 100%|██████████| 10/10 [00:00<00:00, 3070.28 examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2
    })
})


## 🔍 5. Inferência ANTES do fine-tuning (linha de base)

Registramos como o mT5-Large responde **antes** do fine-tuning.

> ⚠️ **Expectativa:** Diferente do Flan-T5, o mT5 **não foi treinado em tarefas de instrução**.
> A resposta antes do fine-tuning tende a ser incoerente ou vazia —
> isso é esperado e normal. O contraste após o fine-tuning será mais marcante.


In [ ]:
def generate_response(model, tokenizer, instruction, max_new_tokens=200):
    """
    Inferência para o mT5-Large.
    Sem prefixo — a entrada vai diretamente para o encoder.
    """
    inputs = tokenizer(
        PREFIX + instruction,   # ← com prefixo
        return_tensors="pt",
        max_length=256,
        truncation=True,
    ).to(model.device)

    with torch.no_grad():
         outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=80,           # ← bem menor que antes
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,      # ← bloqueia repetição de trigramas
            repetition_penalty=1.5,      # ← mais agressivo que o anterior (era 1.3)
            length_penalty=0.6,          # ← favorece respostas ainda mais curtas
            forced_eos_token_id=tokenizer.eos_token_id,  # ← força encerramento
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {TEST_INSTRUCTION}")
print(f"Resposta base: {generate_response(base_model, tokenizer, TEST_INSTRUCTION)}")
print("(Resposta incoerente é esperada — o mT5 não foi treinado em instruções)")


=== ANTES DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta base: 


> **Observação:** A resposta do mT5 antes do fine-tuning tende a ser fragmentada ou vazia.
> Isso é diferente do Flan-T5, que já havia passado por fine-tuning supervisionado.
> O mT5 precisa do fine-tuning para aprender a seguir instruções — é exatamente o que faremos.


## ✂️ 6. Tokenização do Dataset

A tokenização já foi feita dentro da função `convert_to_seq2seq_format` na célula 9,
pois para o Flan-T5 precisamos tokenizar entrada e saída **separadamente e ao mesmo tempo**
para construir o campo `labels` corretamente.

> **Por que `labels` com `-100`?**  
> O Trainer ignora posições com valor `-100` no cálculo da loss.
> Isso é essencial: sem esse mascaramento, o modelo tentaria aprender a prever
> os tokens de padding da saída, o que distorceria o treinamento.


In [ ]:
def tokenize_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=256,
        truncation=True
    )

    cleaned_labels = []
    for label_seq in labels["input_ids"]:
        cleaned_labels.append([
            token if token != tokenizer.pad_token_id else -100 
            for token in label_seq
        ])

    model_inputs["labels"] = cleaned_labels

    return model_inputs

tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenizado:", tokenized_datasets)

Map: 100%|██████████| 2/2 [00:00<00:00, 658.76 examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 2
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa o *gradient checkpointing* e
ajusta as camadas do encoder e do decoder para o treinamento com quantização.


In [ ]:
model = base_model
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar LoRA para o mT5-Large

### `target_modules` para o mT5

O mT5-Large compartilha a mesma arquitetura de atenção do T5 original —
os nomes das camadas lineares são os mesmos: `q`, `k`, `v` e `o`.
Eles aparecem tanto no encoder quanto no decoder.

| Módulo | Onde aparece | Descrição |
|---|---|---|
| `q` | Encoder + Decoder | Projeção das queries |
| `v` | Encoder + Decoder | Projeção dos values |

### `task_type`: `SEQ_2_SEQ_LM`

Informa ao PEFT que o modelo tem arquitetura encoder-decoder,
o que altera como os adaptadores são inseridos e como os gradientes fluem.


In [ ]:
lora_config = LoraConfig(
     r=16,             
    lora_alpha=32,     
    target_modules=["q", "k", "v", "o"],  
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
    inference_mode=False
)

model = get_peft_model(model, lora_config) 
model.print_trainable_parameters()        

trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Seq2Seq

O `DataCollatorForSeq2Seq` é o collator correto para modelos encoder-decoder.
Ele garante que os lotes de `input_ids` e `labels` sejam alinhados corretamente,
respeitando o padding de entrada e saída de forma independente.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `learning_rate` | `3e-4` | Padrão para LoRA em modelos seq2seq |
| `num_train_epochs` | `10` | mT5 parte do zero em instruções — precisa de mais épocas que o Flan-T5 |
| `per_device_train_batch_size` | `8` | mT5-Large é menor — comporta batch maior que o XL |
| `bf16` | `True` | Consistente com o dtype do mT5 |
| `predict_with_generate` | `True` | Avalia gerando texto completo |


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="lora_models/lamini_flant5_neuro",
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=10,
    learning_rate=1e-4,
    per_device_train_batch_size=8,   # ← maior que o XL — modelo menor
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    bf16=True,
    predict_with_generate=True,
    report_to="none",
)


## 🏋️ 11. Inicializar o Trainer

O `Trainer` padrão do Hugging Face funciona para modelos seq2seq quando
`predict_with_generate=True` está configurado no `TrainingArguments`.
Isso faz o Trainer usar o método `generate()` durante a avaliação,
em vez de calcular apenas a loss — o que é mais representativo da qualidade real.


In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    processing_class=tokenizer 
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [12]:
trainer.train()

/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.307000,4.398220
200,0.123200,4.661976


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=200, training_loss=0.7588844114542007, metrics={'train_runtime': 16.0803, 'train_samples_per_second': 49.75, 'train_steps_per_second': 12.438, 'total_flos': 26627958374400.0, 'train_loss': 0.7588844114542007, 'epoch': 100.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Salvamos apenas os adaptadores LoRA — não o modelo mT5 inteiro.
O adaptador com `r=8` e 2 módulos-alvo ocupa tipicamente **5–15 MB**.


In [ ]:
model.save_pretrained("lora_models/lamini_flant5_neuro/final_adapter")
tokenizer.save_pretrained("lora_models/lamini_flant5_neuro/final_tokenizer")

('distilgpt2_tokenizer/tokenizer_config.json',
 'distilgpt2_tokenizer/special_tokens_map.json',
 'distilgpt2_tokenizer/vocab.json',
 'distilgpt2_tokenizer/merges.txt',
 'distilgpt2_tokenizer/added_tokens.json',
 'distilgpt2_tokenizer/tokenizer.json')

## 💻 14. Inferência APÓS o Fine-Tuning

Carregamos o modelo base + adaptador LoRA e comparamos com a resposta anterior.
A função `generate_response` é **idêntica** à usada antes do treino —
no seq2seq, a inferência não muda, pois o encoder sempre recebe só a entrada.


In [ ]:
base_model_eval = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

finetuned_model = PeftModel.from_pretrained(
    base_model_eval,
    "lora_models/lamini_flant5_neuro/final_adapter"
)

finetuned_tokenizer = AutoTokenizer.from_pretrained(
    "lora_models/lamini_flant5_neuro/final_tokenizer"
)

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

print("Modelo fine-tunado carregado com sucesso.")


In [ ]:
print("\n=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {TEST_INSTRUCTION}")

resposta_ajustada = generate_response(
    finetuned_model,
    finetuned_tokenizer,
    TEST_INSTRUCTION
)

print(f"{resposta_ajustada}")


=== DEPOIS DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta ajustada: To use cruise control in a 2023 Subaru Outback:
1. Press the 'CRUISE' button on the steering wheel
2. Accelerate to desired speed (above 25 mph)
3. Press 'SET' to engage


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o mT5-Large produz respostas incoerentes — esperado,
  pois não foi treinado em nenhuma tarefa supervisionada.
- **Depois do fine-tuning:** com LoRA sobre ~200 pares de neurociência, o modelo
  aprende a seguir instruções e responder com terminologia do domínio.

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|---|---|
| **mT5** | Versão multilíngue do T5, pré-treinada em 101 idiomas sem fine-tuning supervisionado. |
| **Sem prefixo de instrução** | O mT5 não foi treinado com prefixos — inserir um não traz benefício. |
| **`AutoTokenizer` com `use_fast=False`** | Forma atual recomendada para o tokenizador SentencePiece do mT5. |
| **`AutoModelForSeq2SeqLM`** | Classe correta para arquiteturas encoder-decoder. |
| **`labels` com `-100`** | Mascara tokens de padding na saída para não entrarem na loss. |
| **`predict_with_generate`** | Avalia gerando texto completo, não só calculando loss. |
| **`task_type=SEQ_2_SEQ_LM`** | Informa ao PEFT que o modelo tem encoder + decoder. |
